In [1]:
# pip install langchain langchain-core langchain-community pypdf pymupdf sentence-transformers chromadb


In [2]:
from langchain_core.documents import Document

In [3]:
sample_doc = Document(
    page_content = "Hello World!",
    metadata = {"source" : "https://www.google.com"}
)

In [4]:
sample_doc

Document(metadata={'source': 'https://www.google.com'}, page_content='Hello World!')

In [5]:
type(sample_doc)

langchain_core.documents.base.Document

In [6]:
# TEXT DATA 
from langchain_community.document_loaders.text import TextLoader

loader = TextLoader("data/Python.txt" , encoding = "utf-8")

C:\Users\mayan\AppData\Local\Temp\ipykernel_16556\4250854095.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders.text import TextLoader


In [7]:
document = loader.load()

In [8]:
# document

In [9]:
# # LOAD PDF DATA 
# from langchain_community.document_loaders.pdf import PyPDFLoader

# pdf_loader = PyPDFLoader("data/research.pdf")
# document = pdf_loader.load()
# document

In [10]:
# # LOAD PDF DATA , PyMuPDFLoader is for complex documents

# from langchain_community.document_loaders.pdf import PyMuPDFLoader

# pdf_loader = PyMuPDFLoader("data/research.pdf")
# document = pdf_loader.load()
# document

# Ingestion Pipeline

## Data ==> Documents 

In [11]:
import os 
from langchain_community.document_loaders.pdf import PyPDFLoader

In [12]:
def load_all_pdfs ():
    folder_path = "data/pdf"
    num_docs = 0
    all_docs = []

    for filename in os.listdir(folder_path):
        if filename.lower().endswith(".pdf"):
            # Complete file path 

            pdf_path = os.path.join(folder_path,filename)

            loader = PyPDFLoader(pdf_path)
            doc = loader.load()

            all_docs.extend(doc)
            num_docs += 1

    print("total Pdf: " , num_docs)
    print("total pages: " , len(all_docs))
    return all_docs

In [13]:
all_pdf_docs = load_all_pdfs()

total Pdf:  2
total pages:  19


## Documents to CHUNKS 

In [14]:
# # CHUNKS 
# pip install langchain_text_splitters

In [15]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

def split_docs(documents , chunk_size = 500, chunk_overlap = 50):

    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size =  chunk_size ,
        chunk_overlap = chunk_overlap
    )

    chunked_doc = text_splitter.split_documents(documents)

    return chunked_doc

In [16]:
chunks = split_docs(all_pdf_docs)

In [17]:
len(chunks)

49

## Converting Chunks to Vector Embeddings

In [18]:
from sentence_transformers import SentenceTransformer

In [19]:
class EmbeddingManager:
    def __init__(self , model_name = "all-MiniLM-L6-v2"):
        
        self.model_name = model_name 
        print("Loading State.....",self.model_name)

        self.model = SentenceTransformer(self.model_name)
        print("embidding Dimensions = " , self.model.get_sentence_embedding_dimension())

    def generate_embeddings(self,text):
        embeddings = self.model.encode(text , show_progress_bar = True)
        print("Embeddings Shape : ",embeddings.shape)
        return embeddings

In [20]:
embedding_manager = EmbeddingManager()

Loading State..... all-MiniLM-L6-v2


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

embidding Dimensions =  384


C:\Users\mayan\AppData\Local\Temp\ipykernel_16556\2567926545.py:8: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print("embidding Dimensions = " , self.model.get_sentence_embedding_dimension())


## Vector Store

In [21]:
import chromadb
import uuid

In [22]:
class VectorStoreManager:
    def __init__(self, persist_directory="data/vector_store", collection_name="pdf_documents"):
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.collection = None
        self.client = None

        self._initialize_store()

    def _initialize_store(self):
        os.makedirs(self.persist_directory, exist_ok=True)
        
        # create a client
        self.client = chromadb.PersistentClient(path=self.persist_directory)

        # create the collection
        self.collection = self.client.get_or_create_collection(
            name=self.collection_name,
            metadata={"description": "vector store collection for pdf embeddings in RAG"}
        )

        print("initialized the vector store with collection:", self.collection_name)
        print("docs in collection:", self.collection.count())

    def add_documents(self, documents, embeddings):
        if len(documents) != len(embeddings):
            raise ValueError("num of documents does not match num of embeddings")


        # store => ids, embedding, document, metadata
        ids = []
        all_metadata = []
        documents_content = []
        embeddings_list = []

        for i, (doc, embedding) in enumerate(zip(documents, embeddings)):
            doc_id = f"doc_{uuid.uuid4()}"
            ids.append(doc_id)

            metadata = dict(doc.metadata)
            metadata["doc_index"] = i
            metadata["content_length"] = len(doc.page_content)
            all_metadata.append(metadata)

            documents_content.append(doc.page_content)

            embeddings_list.append(embedding.tolist())

            self.collection.add(
                ids=ids,
                metadatas=all_metadata,
                documents=documents_content,
                embeddings=embeddings_list
            )

        print("total documents added in vector store=", len(documents_content))
        print("docs in collection:", self.collection.count())

In [23]:
vector_store = VectorStoreManager()

initialized the vector store with collection: pdf_documents
docs in collection: 49


In [24]:
# data => documents => chunks => embeddings => store in vector store

texts = [doc.page_content for doc in chunks]

emebedding = embedding_manager.generate_embeddings(texts)

vector_store.add_documents(chunks, emebedding)

Batches:   0%|          | 0/2 [00:00<?, ?it/s]

Embeddings Shape :  (49, 384)
total documents added in vector store= 49
docs in collection: 98


# Retrieval Pipeline

In [25]:
from sklearn.metrics.pairwise import cosine_similarity

In [26]:
class RAGRetriever:
    def __init__(self, embedding_manager, vector_store):
        self.embedding_manager = embedding_manager
        self.vector_store = vector_store


    def retrieve(self, query, top_k=5, score_threshold=0.0):
        # query => embedding
        query_embeddings = self.embedding_manager.generate_embeddings([query])[0]

        # semantic search
        results = self.vector_store.collection.query(
            query_embeddings=[query_embeddings.tolist()],
            n_results=top_k
        )

        # cosine similarity
        retrieved_docs=[]
        
        if results["documents"] and results["documents"][0]:
            ids = results["ids"][0]
            metadatas = results["metadatas"][0]
            documents = results["documents"][0]
            distances = results["distances"][0]

            for i, (doc_id, metadata, document, distance) in enumerate(zip(ids, metadatas, documents, distances)):
                similarity_score = 1 - distance

                if similarity_score >= score_threshold:
                    retrieved_docs.append({
                        "id": doc_id,
                        "document": document,
                        "metadata": metadata,
                        "distance": distance,
                        "similarity_score": similarity_score,
                        "rank" : i + 1
                    })

            print(f"retrieved {len(retrieved_docs)} documents")

        else:
            print("no documents found")

        return retrieved_docs

In [27]:
rag_retriever = RAGRetriever(embedding_manager, vector_store)

In [28]:
rag_retriever.retrieve("What is encoder decoder")

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Embeddings Shape :  (1, 384)
retrieved 0 documents


[]

## Integrate with LLMs 

### OpenAI - GPT

In [34]:
API_KEY_OPENAI = "API_KEY_OPENAI" 

In [35]:
# pip install langchain_openaiA

In [36]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(
    openai_api_key=API_KEY_OPENAI,
    model="gpt-5.4",
    temperature=0.1,# Defines Creativity of model, low temperature ==> higher facts base 
    max_tokens=1024
)

In [37]:
# generate our retrieval-augmented output
def generate_output(query, retriever, llm, top_k=3):
    results = retriever.retrieve(query, top_k)

    context = "\n".join([doc["document"] for doc in results]) if results else ""

    if not context:
        print("we found no relevant context for the given query")

    # context + query
    prompt = f""" use given context to generate the answer for the query
                Context: {context}
                Query: {query} """

    response = llm.invoke(prompt) # expecting a string as prompt
    return response.content

In [39]:
# answer = generate_output("what is Histogram?", rag_retriever, llm)

### Groq

In [40]:
API_Key_GROQ = "paste-api-key-here"

In [42]:
# pip install langchain-groq

In [43]:
from langchain_groq import ChatGroq

llm = ChatGroq(
    groq_api_key=API_Key_GROQ,
    model="qwen/qwen3-32b",
    temperature=0.1,
    max_tokens=1024
)

In [44]:
# generate our retrieval-augmented output
def generate_output(query, retriever, llm, top_k=3):
    results = retriever.retrieve(query, top_k)

    context = "\n".join([doc["document"] for doc in results]) if results else ""

    if not context:
        print("we found no relevant context for the given query")

    # context + query
    prompt = f""" use given context to generate the answer for the query
                Context: {context}
                Query: {query} """

    response = llm.invoke([prompt.format(context=context, query=query)]) # expecting a list as prompt
    return response.content

In [ ]:
answer = generate_output("what is RAG?", rag_retriever, llm)

In [ ]:
print(answer)